Our Project
This should be our main commit file for the profs to go through

In [ ]:
import Evaluation
import cleaning
import pickle
import pandas as pd
import cleaning
import networkx as nx

Data cleaning

In [ ]:
file_path = "publicdataexportv131450706334_with_lon_lat.xlsx"

In [ ]:
# Loading sheets
bus = pd.read_excel(file_path, sheet_name="Bus", header=3)
line = pd.read_excel(file_path, sheet_name="Line", header=3)
gen = pd.read_excel(file_path, sheet_name="Generator", header=3)
load = pd.read_excel(file_path, sheet_name="Load", header=3)
hvdc = pd.read_excel(file_path, sheet_name="HVDC", header=3)
transformer2 = pd.read_excel(file_path, sheet_name="Transformer2", header=3)
transformer3 = pd.read_excel(file_path, sheet_name="Transformer3", header=3)

for df in [bus, line, gen, load, hvdc, transformer2, transformer3]:
    df.columns = [str(c).strip() for c in df.columns]


# This is our main cleaning function that turns our raw data into an unaggregated graph, that is used for the scenario generation
g_unaggregated = cleaning.main_clean(bus, line, gen, load, hvdc, transformer2, transformer3)
edges = pd.read_excel('danish_grid_graph_ready.xlsx', sheet_name='edges')
nodes = pd.read_excel('danish_grid_graph_ready.xlsx', sheet_name='nodes')

In [ ]:
cleaning.print_graph(cleaning.aggregate_graph(g_unaggregated, edges))

## Scenario-conditioned attack simulations

Only balanced scenarios (supply ≈ demand after rebalancing) are valid baselines for attack evaluation.
If the grid already has unmet demand before an attack, the simulation results cannot isolate the attack's impact.

In [ ]:
base_scenario = g_unaggregated

In [ ]:
from scenarios import SCENARIOS, apply_scenario_mean

# ── Check balance for every scenario ─────────────────────────────────────────
BALANCE_TOL_PCT = 1.0   # allow up to 1 % residual gap

balance_rows = []
for sid, scenario in SCENARIOS.items():
    G_sc = apply_scenario_mean(base_scenario, scenario)
    total_supply = sum(d.get('supply', 0.0) for _, d in G_sc.nodes(data=True))
    total_demand = sum(d.get('demand', 0.0) for _, d in G_sc.nodes(data=True))
    gap          = abs(total_supply - total_demand)
    gap_pct      = gap / max(total_demand, 1.0) * 100
    balance_rows.append({
        'scenario_id'     : sid,
        'label'           : scenario['label'],
        'total_supply_MW' : round(total_supply, 1),
        'total_demand_MW' : round(total_demand, 1),
        'gap_MW'          : round(gap, 1),
        'gap_pct'         : round(gap_pct, 3),
        'balanced'        : gap_pct <= BALANCE_TOL_PCT,
    })

balance_df = pd.DataFrame(balance_rows)
balance_df

In [ ]:
balanced_ids = balance_df.loc[balance_df['balanced'], 'scenario_id'].tolist()
skipped_ids  = balance_df.loc[~balance_df['balanced'], 'scenario_id'].tolist()

print(f"Balanced ({len(balanced_ids)}): {balanced_ids}")
if skipped_ids:
    print(f"Skipped  ({len(skipped_ids)}): {skipped_ids}")

# ── Run single-removal attack simulation for each balanced scenario ──────────
scenario_results_list = []
for sid in balanced_ids:
    scenario = SCENARIOS[sid]
    G_sc = apply_scenario_mean(base_scenario, scenario)
    G_sc.name = sid
    print(f"\n→ {sid}: {scenario['label']}")
    df = Evaluation.simulation_all_single_removals(cleaning.aggregate_graph(G_sc, edges))
    df.insert(0, 'scenario_id',    sid)
    df.insert(1, 'scenario_label', scenario['label'])
    scenario_results_list.append(df)

scenario_results = pd.concat(scenario_results_list, ignore_index=True)
scenario_results.to_excel("scenario_results.xlsx", index=False)
print(f"\nSaved {len(scenario_results)} rows to scenario_results.xlsx")

In [ ]:
scenario_results.sort_values('total_unmet_demand_MW', ascending=False).head(20)